# BF16 GEMM on A100: SGEMM_CUDA 12단계를 Triton으로

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/triton_gemm_bf16_a100.ipynb)

[siboehm/SGEMM_CUDA](https://github.com/siboehm/SGEMM_CUDA)는 FP32 행렬곱 커널을 CUDA로 12단계에 걸쳐 최적화한다. 이 노트북은 같은 최적화 흐름을 **Triton**으로 다시 따라간다. 대상은 **A100 Tensor Core 위의 BF16 GEMM**이다(입력 bf16, 누산 fp32, 출력 bf16).

CUDA에서 손으로 하던 일 상당수는 Triton 컴파일러가 대신한다. 그래서 12개 커널을 1:1로 옮기지 않고, **사람이 정해야 하는 결정 단위로 6개 버전**을 만든다.

| 버전 | 바꾸는 것 | SGEMM_CUDA 대응 |
|---|---|---|
| v1 | `tl.dot` 없이 k를 하나씩 도는 outer product (CUDA core FMA) | K1·K2·K5 |
| v2 | `tl.dot` 64×64×32 타일, 파이프라인 없음 (`num_stages=1`) | K3 smem tiling, K10 warptiling, + Tensor Core |
| v3 | v2 + `num_stages=4` | K11·K12 double buffering → A100 `cp.async` multistage |
| v4 | 타일 128×128×64, `num_warps=8` | K5→K9 타일 키우기 |
| v5 | v4 + `GROUP_SIZE_M=8` (L2 친화 블록 순서) | K4 주석의 blockIdx 순서 |
| v6 | `@triton.autotune` | K9 autotuner |

K2(coalescing), K6(float4 벡터화), K7·K8(bank conflict 회피)에 해당하는 버전은 따로 없다. **컴파일러가 자동으로 처리**하기 때문이다. 4절에서 PTX/SASS를 열어 실제로 그렇게 내려갔는지 확인한다.

**런타임 설정**: 메뉴 `런타임 → 런타임 유형 변경 → A100 GPU`. 다른 GPU에서도 돌아가지만 peak 대비 비율과 PTX 명령은 달라진다.

## 0. 환경 확인

In [ ]:
import torch, triton
import triton.language as tl
assert torch.cuda.is_available(), "런타임 유형을 GPU로 바꾸세요"
p = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| triton", triton.__version__)
print(p.name, "| SMs", p.multi_processor_count, "| HBM", round(p.total_memory / 2**30, 1), "GiB | cc", f"{p.major}.{p.minor}")
if "A100" not in p.name:
    print("⚠️ A100이 아닙니다. 돌아가긴 하지만 peak 대비 비율과 PTX 명령이 달라집니다.")

## 1. 공통 하네스

- 입력: `a ~ N(0,1)`, `b ~ N(0,1)/√K`. 출력 원소의 크기가 1 근처에 머물러서 크기와 상관없이 같은 허용 오차를 쓸 수 있다.
- 정답: `a.float() @ b.float()` (FP32). bf16 출력의 반올림 오차를 감안해 `atol = rtol = 2e-2`로 비교한다.
- 시간: `triton.testing.do_bench`의 median. 측정 사이마다 L2를 비운다.
- TFLOPs = 2·M·N·K / 시간. 기준은 A100 BF16 dense peak **312 TFLOPs**와 cuBLAS(`torch.matmul`)다.

In [ ]:
import math
import pandas as pd

PEAK_TFLOPS = 312.0          # A100 BF16 Tensor Core, dense
ATOL = RTOL = 2e-2


def make_inputs(M, N, K, seed=0):
    g = torch.Generator(device="cuda").manual_seed(seed)
    a = torch.randn(M, K, device="cuda", generator=g).to(torch.bfloat16)
    b = (torch.randn(K, N, device="cuda", generator=g) / math.sqrt(K)).to(torch.bfloat16)
    return a, b


def check(fn, M, N, K):
    a, b = make_inputs(M, N, K, seed=M + 7 * N + 13 * K)
    c = fn(a, b)
    ref = a.float() @ b.float()
    assert c.shape == (M, N) and c.dtype == torch.bfloat16, (c.shape, c.dtype)
    err = (c.float() - ref).abs().max().item()
    ok = torch.allclose(c.float(), ref, atol=ATOL, rtol=RTOL)
    return ok, err


def bench(fn, M, N, K):
    a, b = make_inputs(M, N, K)
    fn(a, b)                                     # 컴파일·autotune은 측정에서 뺀다
    ms = triton.testing.do_bench(lambda: fn(a, b), return_mode="median")
    return ms, 2 * M * N * K / (ms * 1e-3) / 1e12


def cublas(a, b):
    return torch.matmul(a, b)

## 2. 버전별 커널

### v1. outer product: `tl.dot` 없이 CUDA core로

프로그램 하나가 출력 타일 `BLOCK_M × BLOCK_N`을 맡는다. k를 **하나씩** 돌면서 A의 열 하나(길이 BLOCK_M)와 B의 행 하나(길이 BLOCK_N)를 읽고, 외적을 누산기에 더한다. SGEMM_CUDA K5의 register blocking과 같은 계산이다.

- `tl.dot`을 쓰지 않으므로 Tensor Core가 아니라 CUDA core의 FP32 FMA로 계산한다.
- A의 열은 메모리에서 K 간격으로 떨어져 있다. 그래서 K1처럼 coalescing되지 않는 접근이 된다.
- 입력 조각을 shared memory에 올려 재사용하는 구조가 없다. k마다 L1/L2에서 다시 읽는다.

In [ ]:
@triton.jit
def outer_product_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask_m = offs_m < M
    mask_n = offs_n < N
    a_ptrs = a_ptr + offs_m * stride_am          # A[:, 0]
    b_ptrs = b_ptr + offs_n * stride_bn          # B[0, :]

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K):
        a = tl.load(a_ptrs, mask=mask_m, other=0.0).to(tl.float32)
        b = tl.load(b_ptrs, mask=mask_n, other=0.0).to(tl.float32)
        acc += a[:, None] * b[None, :]           # rank-1 update = FFMA
        a_ptrs += stride_ak
        b_ptrs += stride_bk

    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc.to(tl.bfloat16), mask=mask_m[:, None] & mask_n[None, :])


V1 = dict(BLOCK_M=64, BLOCK_N=64, num_warps=4)


def launch_v1(a, b):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.bfloat16)
    grid = (triton.cdiv(M, V1["BLOCK_M"]), triton.cdiv(N, V1["BLOCK_N"]))
    handle = outer_product_kernel[grid](
        a, b, c, M, N, K, *a.stride(), *b.stride(), *c.stride(), **V1)
    return c, handle


def gemm_v1(a, b):
    return launch_v1(a, b)[0]

### v2~v5. 타일 GEMM 커널 하나, 파라미터만 바꾼다

v2부터 v5까지는 **커널 코드가 하나**다. 바뀌는 것은 launch 파라미터뿐이다. Triton에서 최적화한다는 것은 대부분 이 파라미터를 고르는 일이다.

커널 본문은 4주차 [Tiling](../week4/03-tiling.md)의 블록 알고리즘을 그대로 옮긴 것이다. 출력 타일을 고정하고, K 방향으로 A·B 조각을 바꿔 가며 `tl.dot`으로 누산한다. 경계는 `mask`로 처리한다. 그래서 SGEMM_CUDA runner의 "M<128이면 64 타일" 같은 우회 없이 아무 크기에서나 동작한다.

`tl.dot`이 컴파일러에 넘기는 일은 다음과 같다.

| CUDA에서 직접 하던 일 | 해당 커널 | Triton에서 |
|---|---|---|
| A·B 조각을 shared memory에 올리기 | K3 | `tl.dot` 피연산자를 컴파일러가 smem에 배치 |
| warp/thread별 출력 분할 | K5·K10 | `num_warps`에 맞춰 자동 분할 |
| coalescing, 128-bit 로드 | K2·K6 | 포인터 연속성을 분석해 자동 벡터화 |
| bank conflict 회피 | K7·K8 | smem swizzle 자동 적용 |
| 연산 명령 | FFMA | bf16 `mma.sync` (Tensor Core) + `ldmatrix` |

**블록 순서(`GROUP_SIZE_M`)**: 1차원 program id를 출력 타일 (pid_m, pid_n)에 대응시키는 방식이다. `GROUP_SIZE_M=1`이면 행 우선으로 타일을 돈다. 8이면 행 8개 묶음 안에서 열 방향으로 지그재그로 돈다. 동시에 실행되는 프로그램들이 같은 A 행 블록과 B 열 블록을 공유하게 되어 L2 적중률이 올라간다. SGEMM_CUDA K4 주석에 적힌 "blockIdx 순서를 바꾸면 30% 차이" 문제와 같은 원리다.

In [ ]:
@triton.jit
def gemm_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    # program id -> 출력 타일 (pid_m, pid_n). GROUP_SIZE_M=1이면 행 우선 순서.
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        k_left = K - k * BLOCK_K
        a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (offs_k[None, :] < k_left), other=0.0)
        b = tl.load(b_ptrs, mask=(offs_k[:, None] < k_left) & (offs_n[None, :] < N), other=0.0)
        acc = tl.dot(a, b, acc)                  # bf16 x bf16 -> fp32, Tensor Core
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc.to(tl.bfloat16), mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))


def launch_gemm(a, b, cfg, kernel=gemm_kernel):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.bfloat16)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]) * triton.cdiv(N, meta["BLOCK_N"]),)
    handle = kernel[grid](a, b, c, M, N, K, *a.stride(), *b.stride(), *c.stride(), **cfg)
    return c, handle

| 버전 | BLOCK_M×N×K | num_warps | num_stages | GROUP_SIZE_M | 의도 |
|---|---|---|---|---|---|
| v2 | 64×64×32 | 4 | 1 | 1 | Tensor Core만, 로드와 연산이 겹치지 않음 |
| v3 | 64×64×32 | 4 | 4 | 1 | 다음 K 조각을 미리 `cp.async`로 받음 |
| v4 | 128×128×64 | 8 | 3 | 1 | 타일을 키워 산술 강도↑ (smem 3×32 KB = 96 KB) |
| v5 | 128×128×64 | 8 | 3 | 8 | 블록 순서만 바꿔 L2 재사용↑ |

`num_stages`는 K 루프의 소프트웨어 파이프라인 깊이다. A100에서는 다음 조각을 `cp.async`로 smem에 미리 받아 두는 multistage 구조가 된다. SGEMM_CUDA K11은 스레드를 반씩 나눠 같은 효과를 노렸고, K12는 `cuda::memcpy_async`로 2단을 만들었다. 여기서는 숫자 하나만 바꾸면 된다. 대신 stage마다 smem을 `(BM·BK + BK·BN)·2 B`씩 더 쓰므로 occupancy와 맞바꾸는 셈이다(4주차 [타일 크기와 Occupancy](../week4/04-tile-size-and-occupancy.md)).

In [ ]:
V2 = dict(BLOCK_M=64,  BLOCK_N=64,  BLOCK_K=32, GROUP_SIZE_M=1, num_warps=4, num_stages=1)
V3 = dict(BLOCK_M=64,  BLOCK_N=64,  BLOCK_K=32, GROUP_SIZE_M=1, num_warps=4, num_stages=4)
V4 = dict(BLOCK_M=128, BLOCK_N=128, BLOCK_K=64, GROUP_SIZE_M=1, num_warps=8, num_stages=3)
V5 = dict(V4, GROUP_SIZE_M=8)


def gemm_v2(a, b): return launch_gemm(a, b, V2)[0]
def gemm_v3(a, b): return launch_gemm(a, b, V3)[0]
def gemm_v4(a, b): return launch_gemm(a, b, V4)[0]
def gemm_v5(a, b): return launch_gemm(a, b, V5)[0]

### v6. autotune: K9의 셸 스크립트를 데코레이터로

SGEMM_CUDA는 `scripts/kernel_9_autotuner.sh`로 파라미터를 바꿔 가며 다시 빌드하고 측정했다. Triton은 후보 config를 나열하면 **처음 호출될 때 GPU 위에서 전부 측정하고** 가장 빠른 것을 `key`(여기서는 M, N, K)별로 캐시한다.

후보는 Triton matmul 튜토리얼의 CUDA용 목록에 A100 큰 타일 몇 개를 더했다. smem 한도(블록당 최대 약 163 KB)를 넘는 조합은 넣지 않았다. 예를 들어 128×256×64를 3 stage로 돌리면 3 × (128·64 + 64·256) × 2 B = 144 KB다.

In [ ]:
def _cfg(bm, bn, bk, stages, warps):
    return triton.Config(dict(BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk, GROUP_SIZE_M=8),
                         num_stages=stages, num_warps=warps)


AUTOTUNE_CONFIGS = [
    _cfg(128, 256, 64, 3, 8), _cfg(256, 128, 64, 3, 8), _cfg(128, 128, 64, 4, 4),
    _cfg(128, 128, 64, 3, 8), _cfg(128, 128, 32, 4, 4), _cfg(64, 256, 32, 4, 4),
    _cfg(128, 64, 32, 4, 4), _cfg(64, 128, 32, 4, 4), _cfg(128, 32, 32, 4, 4),
    _cfg(64, 32, 32, 5, 2), _cfg(32, 64, 32, 5, 2),
]

# 같은 gemm_kernel을 autotuner로 감싼다. 커널 코드는 그대로다.
gemm_kernel_autotuned = triton.autotune(configs=AUTOTUNE_CONFIGS, key=["M", "N", "K"])(gemm_kernel)


def gemm_v6(a, b):
    return launch_gemm(a, b, {}, kernel=gemm_kernel_autotuned)[0]


def best_config_v6():
    best = gemm_kernel_autotuned.best_config
    return dict(best.kwargs, num_warps=best.num_warps, num_stages=best.num_stages)

### A100에서 쓸 수 없는 것: TMA, wgmma, warp specialization

Hopper(sm_90) 이후 Triton matmul은 `tl.make_tensor_descriptor`로 **TMA**를 쓰고, `tl.dot`을 **wgmma**로 내리고, `tl.range(..., warp_specialize=True)`로 로드 warp와 연산 warp를 나눈다. A100(sm_80)에는 이 하드웨어가 없다. 그래서 이 노트북의 최종형은 **`cp.async` multistage + `ldmatrix` + `mma.sync`**다. 같은 커널을 H100에서 돌리면 `tl.dot`은 자동으로 wgmma가 된다. 다만 TMA와 warp specialization은 descriptor와 옵션을 명시해야 켜진다.

## 3. 정확성 검증

정사각 크기와 타일 배수가 아닌 크기를 섞어 확인한다. v6은 shape마다 처음 호출될 때 autotune이 돌기 때문에 이 셀이 조금 오래 걸린다.

In [ ]:
VERSIONS = {"v1": gemm_v1, "v2": gemm_v2, "v3": gemm_v3, "v4": gemm_v4,
            "v5": gemm_v5, "v6": gemm_v6, "cuBLAS": cublas}
CHECK_SHAPES = [(512, 512, 512), (1000, 1000, 1000), (4096, 1024, 2048), (333, 777, 129)]

rows = []
for name, fn in VERSIONS.items():
    for M, N, K in CHECK_SHAPES:
        ok, err = check(fn, M, N, K)
        rows.append(dict(version=name, shape=f"{M}x{N}x{K}", ok=ok, max_abs_err=round(err, 5)))
check_df = pd.DataFrame(rows)
display(check_df.pivot(index="version", columns="shape", values="max_abs_err").loc[list(VERSIONS)])
assert check_df.ok.all(), check_df[~check_df.ok]
print("모든 버전·shape 통과")

## 4. 컴파일러가 실제로 무엇을 만들었나: PTX와 SASS

`kernel[grid](...)`는 컴파일된 커널 핸들을 돌려준다. `handle.asm["ptx"]`에서 핵심 명령의 **정적 개수**(코드에 등장하는 횟수, 실행 횟수 아님)를 센다.

| 명령 | 뜻 | 관련 커널 |
|---|---|---|
| `fma.rn.f32` | CUDA core FP32 FMA | SGEMM_CUDA 전체 |
| `mma.sync` (SASS `HMMA`) | warp 단위 Tensor Core 행렬곱 | 없음 (새로 생긴 것) |
| `ldmatrix` (SASS `LDSM`) | smem → mma fragment 로드 | K5·K10의 regM/regN 로드 |
| `cp.async` (SASS `LDGSTS`) | GMEM → SMEM 비동기 복사 | K11·K12 |
| `ld.global.v4` / `.v2` | 벡터 로드 | K6 float4 |
| `bar.sync` | `__syncthreads()` | 전체 |

In [ ]:
import re

PTX_PATTERNS = {
    "fma.rn.f32": r"\bfma\.rn\.f32\b",
    "mma.sync": r"\bmma\.sync\b",
    "ldmatrix": r"\bldmatrix\b",
    "cp.async": r"\bcp\.async\.c[ag]\b",
    "ld.global.v4/v2": r"\bld\.global\S*\.v[24]\b",
    "bar.sync": r"\bbar\.sync\b",
}
SASS_PATTERNS = {"FFMA": r"\bFFMA\b", "HMMA": r"\bHMMA\b", "LDSM": r"\bLDSM\b", "LDGSTS": r"\bLDGSTS\b"}

a, b = make_inputs(1024, 1024, 1024)
gemm_v6(a, b)                                     # autotune 결과 확보
handles = {
    "v1": launch_v1(a, b)[1],
    "v2": launch_gemm(a, b, V2)[1],
    "v3": launch_gemm(a, b, V3)[1],
    "v5": launch_gemm(a, b, V5)[1],
    "v6": launch_gemm(a, b, best_config_v6())[1],
}

rows = []
for name, h in handles.items():
    ptx = h.asm["ptx"]
    row = {"version": name, **{k: len(re.findall(p, ptx)) for k, p in PTX_PATTERNS.items()}}
    try:
        sass = h.asm["sass"]
        row.update({k: len(re.findall(p, sass)) for k, p in SASS_PATTERNS.items()})
    except Exception as e:                        # nvdisasm이 없으면 SASS는 건너뜀
        row["sass"] = f"n/a ({type(e).__name__})"
    row["smem_bytes"] = h.metadata.shared
    row["regs"] = getattr(h, "n_regs", None)
    rows.append(row)
ptx_df = pd.DataFrame(rows).set_index("version")
display(ptx_df)
print("v6 best config @1024³:", best_config_v6())

**읽는 법**

- **v1**: `fma.rn.f32`만 있고 `mma.sync`가 0이면 CUDA core로만 계산한 것이다.
- **v2**: `mma.sync`와 `ldmatrix`가 생긴다. `tl.dot` 한 줄이 Tensor Core 경로로 내려갔다는 뜻이다. `cp.async`는 아직 0이어야 한다. `num_stages=1`이라 파이프라인이 없기 때문이다.
- **v3 이후**: `cp.async`가 나타난다. SGEMM_CUDA K12의 `memcpy_async`가 A100에서 실제로 내려가는 명령이 이것이다.
- **smem_bytes**: stage 수와 타일 크기에 비례해 커진다. 레지스터 수(`regs`)와 함께 occupancy를 결정한다.

v2의 Triton 코드에는 swizzle이나 bank conflict라는 말이 한 번도 나오지 않는다. 그런데도 `ldmatrix`가 conflict 없이 읽을 수 있도록 컴파일러가 smem 레이아웃을 정했다. 직접 확인하고 싶으면 `handles["v3"].asm["ttgir"]`에서 `#shared` 레이아웃의 `vec`/`perPhase`/`maxPhase` 값을 찾아보면 된다. 이 세 값이 XOR swizzle 파라미터다.

In [ ]:
for line in handles["v3"].asm["ttgir"].splitlines():
    if line.startswith("#shared"):
        print(line)

## 5. 벤치마크

SGEMM_CUDA와 같은 정사각 크기(128~4096)에 8192를 더해 측정한다. v1은 큰 크기에서 오래 걸리므로 4096까지만 잰다.

In [ ]:
SIZES = [128, 256, 512, 1024, 2048, 4096, 8192]
V1_MAX = 4096

rows = []
for n in SIZES:
    for name, fn in VERSIONS.items():
        if name == "v1" and n > V1_MAX:
            continue
        ms, tflops = bench(fn, n, n, n)
        rows.append(dict(size=n, version=name, ms=ms, tflops=tflops))
    print(f"{n:>5} done")
bench_df = pd.DataFrame(rows)
cub = bench_df[bench_df.version == "cuBLAS"].set_index("size").tflops
bench_df["pct_cublas"] = 100 * bench_df.tflops / bench_df["size"].map(cub)
bench_df["pct_peak"] = 100 * bench_df.tflops / PEAK_TFLOPS

order = list(VERSIONS)
display(bench_df.pivot(index="version", columns="size", values="tflops").loc[order].round(1))

4096³에서 버전별로 정리하면 다음과 같다(SGEMM_CUDA README 표와 같은 형식).

In [ ]:
t = bench_df[bench_df["size"] == 4096].set_index("version").loc[order]
display(t[["tflops", "pct_cublas", "pct_peak"]].round(1).rename(
    columns={"tflops": "TFLOPs/s", "pct_cublas": "% of cuBLAS", "pct_peak": "% of 312 peak"}))

In [ ]:
import matplotlib.pyplot as plt

# 범주색은 고정 순서로 부여하고, cuBLAS는 기준선이라 무채색 점선으로 그린다.
COLORS = {"v1": "#2a78d6", "v2": "#eb6834", "v3": "#1baf7a", "v4": "#eda100",
          "v5": "#e87ba4", "v6": "#008300", "cuBLAS": "#52514e"}

fig, ax = plt.subplots(figsize=(8, 4.8))
for name in order:
    d = bench_df[bench_df.version == name].sort_values("size")
    ax.plot(d["size"], d.tflops, marker="o", markersize=5, linewidth=2, color=COLORS[name],
            linestyle="--" if name == "cuBLAS" else "-", label=name)
ax.axhline(PEAK_TFLOPS, color="#9a9893", linewidth=1, linestyle=":")
ax.text(SIZES[0], PEAK_TFLOPS, " BF16 peak 312", va="bottom", fontsize=9, color="#52514e")
ax.set_xscale("log", base=2)
ax.set_xticks(SIZES, [str(s) for s in SIZES])
ax.set_xlabel("M = N = K")
ax.set_ylabel("TFLOPs/s")
ax.set_title("Triton BF16 GEMM on A100: v1 → v6 vs cuBLAS")
ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, ncol=4, loc="upper left", bbox_to_anchor=(0, -0.15))
fig.tight_layout()
plt.show()

## 6. 정리

아래 수치는 A100-SXM4-40GB에서 한 번 실행한 결과다(Triton 3.6, torch 2.11). 실행할 때마다 몇 % 정도 달라질 수 있다.

| 4096³ | TFLOPs/s | cuBLAS 대비 | 직전 대비 |
|---|---|---|---|
| v1 outer product | 2.4 | 0.9% | |
| v2 `tl.dot` | 108.7 | 42% | ×45 |
| v3 + `num_stages=4` | 134.1 | 52% | +23% |
| v4 128×128×64 타일 | 180.4 | 70% | +35% |
| v5 + `GROUP_SIZE_M=8` | 182.6 | 71% | +1% |
| v6 autotune | 220.8 | 86% | +21% |
| cuBLAS | 257.1 | 100% | |

- **v1 → v2 (×45)**: 계산 단위 자체가 바뀌는 단계다. v1의 2.4 TFLOPs는 FP32 CUDA core peak(19.5)의 12% 수준이다. coalescing되지 않는 A 열 접근이 발목을 잡는다. `tl.dot` 한 줄로 `mma.sync`와 `ldmatrix`가 생기면서 상한이 312 TFLOPs로 올라간다. SGEMM_CUDA가 K1부터 K10까지 12단계에 걸쳐 얻은 것보다 큰 폭이다.
- **v2 → v3 (+23%)**: `cp.async`(SASS `LDGSTS`)가 생긴다. FP32 때는 K11 double buffering이 K10보다 오히려 느렸다. 하지만 Tensor Core는 연산이 빨라서 로드 지연이 그대로 드러나고, multistage가 실제로 효과를 낸다.
- **v3 → v4 (+35%)**: 타일을 64²에서 128²로 키우면 HBM과 smem에서 읽는 양 대비 연산량이 두 배가 된다(4주차 05 손계산과 같은 논리). 8192에서 차이가 더 분명하다. v2와 v3는 80~85로 떨어지지만 v4는 180을 유지한다. 작은 타일은 같은 입력을 더 여러 번 읽어서 큰 행렬에서 L2/HBM 대역폭에 걸린다.
- **v4 → v5 (4096 +1%, 8192 +6%)**: 커널 내부는 그대로 두고 블록 순서만 바꿨다. 행렬이 클수록 L2에 다 들어가지 않아서 순서 효과가 커진다.
- **v6 (+21%)**: 크기마다 최적점이 다르다. 1024³에서는 64×128×32가 뽑혔다. 128×128 타일이면 출력 타일이 64개뿐이라 108개 SM을 다 채우지 못하기 때문이다. 같은 이유로 1024에서 v4(77.7)가 v3(83.9)보다 느리다. 2048에서는 v6(182)이 cuBLAS(154)보다 빨랐다.
- **SGEMM_CUDA와의 비교**: README의 최종 결과는 A6000 FP32 SIMT에서 21.8 TFLOPs(cuBLAS 대비 94%)다. 여기서는 BF16 Tensor Core라 절대값은 10배 크다. 그러나 cuBLAS 대비 비율은 86%로 더 낮다. 남은 차이를 줄이려면 split-K, epilogue 최적화, 더 정교한 config 탐색이 필요하다. Hopper라면 TMA와 wgmma가 다음 단계가 된다.